In [5]:
import requests
import json
import os
import sys
sys.path.append("../src")

from config import FREECRYPTO_TOKEN,NEWS_API_KEY,GROQ_API_KEY
from langchain_core.tools import tool
from newsapi import NewsApiClient
from langchain_groq import ChatGroq

In [7]:
llm = ChatGroq(model="qwen/qwen3.8-27b",api_key=GROQ_API_KEY)

In [10]:
@tool
def crypto_list_tool(limit:int=25)-> str:
    """
    Fetches a list of available cryptocurrencies (symbol + name) from FreeCryptoAPI.
    Args:
        limit (int): Max number of coins to return (to keep responses compact).
    Returns:
        str: JSON string with condensed list: [{"symbol": "...","name":"..."},...]
    """

    url = "https://api.freecryptoapi.com/v1/getCryptoList"
    headers = {"Authorization":f"Bearer {FREECRYPTO_TOKEN}"}
    resp = requests.get(url,headers=headers,timeout=30)
    try:
        data = resp.json()
    except Exception:
        return json.dumps({"status":False,"error":"Invalid JSON from API"})
    
    if isinstance(data,list):
        slim = []
        for item in data[:max(1,limit)]:
            sym = item.get("symbol") or item.get("symbol") or item.get("ticker")
            nm = item.get("name") or item.get("Name")
            if sym or nm:
                slim.append({"symbol":sym,"name":nm})
        return json.dumps({"status":True,"coins":slim},indent=2)
    return json.dumps(data,indent=2)

In [11]:
#crypto data tool for one Cryptocurrency :fetch detialed data for a single symbol
@tool 
def crypto_data_too(symbol) -> str :
    """
    Fetches detailed data for a given crypto symbol (e.g.,BTC) from FreeCryptoAPI.
    Args:
        - symbol (str): Ticker symbol like 'BTC','ETH', etc.
    Returns:
    str: Json string ot the API response.
    """
    url="https://api.freecryptoapi.com/v1/getData"
    headers = {"Authorization":f"Bearer {FREECRYPTO_TOKEN}"}
    params = {"symbol":symbol.upper().strip()}
    resp = requests.get(url,headers=headers,params=params)

    try:
        data = resp.json()
    except Exception:
        return json.dumps({"status":False,"error":"Invalid JSON from API"})

    return json.dumps(data,indent=2)



In [13]:
#crypto news via News api
@tool
def crypto_news_tool(query:str="crypto",mode:str="top",max_items:int=5)->str:
    """
    Fetches recent/important news using NewsApi and returns a condensed list.
    Args:
        - query(str): Search query, eg 'bitcoin',''ethereum','crypto regulation'
        - max items (int): Max number of articles to return (default: 5)
    Returns:
        str: Json string with fields : title, source , url,publishedAt
    """

    newsapi = NewsApiClient(api_key=NEWS_API_KEY)

    try: 
        res = newsapi.get_everything(
            q = query,
            language = "en",
            sort_by="publishedAt",
            page=1
        )
    except Exception as e:
        return json.dumps({"status":False,"error":str(e)})
    
    slim: List[Dict[str,Any]] =[]
    for a in articles[:max(1,max_items)]:
        slim.append({
            "title":a.get("title"),
            "source":(a.get("source") or {}).get("name"),
            "url": a.get("publishedAt"),
            "description": a.get("description"),
            "content": a.get("content")
        })

    return json.dumps(
        {
            "status":True,
            "totalResults":len(articles),
            "returned":len(slim),
            "articles":slim,
        },
        indent=2
    )

In [2]:
%pip install langchain-groq


Note: you may need to restart the kernel to use updated packages.
